# Plot Retrieval Results

Set the paths and round in the first cell, then run the notebook top to bottom.


In [ ]:
from pathlib import Path
from io import StringIO

import numpy as np
import matplotlib.pyplot as plt
from corner import corner

from floppity import Retrieval, RetrievalOutput, helpers

OUTPUT_DIR = Path("/Users/floppityflappity/Work/WISE1828/test_codex_version_3")
RETRIEVAL_PATH = OUTPUT_DIR / "retrieval.pkl"
ROUND = "latest"  # use "latest" or an integer round index, e.g. 0, 1, 2
N_CORNER_SAMPLES = 5000


In [ ]:
def available_rounds(output_dir):
    round_root = output_dir / "rounds"
    round_dirs = sorted(round_root.glob("round_*"))
    rounds = []
    for path in round_dirs:
        try:
            rounds.append(int(path.name.split("_")[-1]))
        except ValueError:
            continue
    return rounds


def resolve_round(output_dir, selected_round):
    rounds = available_rounds(output_dir)
    if not rounds:
        raise FileNotFoundError(f"No round directories found in {output_dir / 'rounds'}")
    if selected_round == "latest":
        return rounds[-1]
    if selected_round not in rounds:
        raise FileNotFoundError(
            f"Round {selected_round} is not available. Available rounds: {rounds}"
        )
    return selected_round


def round_data_path(output_dir, round_index):
    return output_dir / "rounds" / f"round_{round_index:03d}" / "training_data.npz"


def atmosphere_path(output_dir, round_index):
    path = output_dir / f"mixingratios_round_{round_index}.dat"
    if not path.exists():
        raise FileNotFoundError(f"No atmosphere file found for round {round_index}: {path}")
    return path


def proposal_for_corner(retrieval, alpha):
    if alpha > 0:
        if getattr(retrieval, "posteriors", None):
            return retrieval.posteriors[-1]
        proposal = retrieval.proposals[-1]
        if hasattr(proposal, "posterior"):
            return proposal.posterior
    return retrieval.proposals[-1]


def proposal_sample_mask(round_data, alpha):
    sources = round_data.get("sample_sources")
    if alpha <= 0:
        return slice(None)
    if sources is None:
        raise ValueError(
            "This run used alpha > 0, but the round archive has no sample_sources. "
            "Re-run with save_data=True using the current FlopPITy branch."
        )
    return sources == "proposal"


def sigma_bands(values):
    return {
        "3sigma": np.nanpercentile(values, [0.135, 99.865], axis=0),
        "2sigma": np.nanpercentile(values, [2.275, 97.725], axis=0),
        "1sigma": np.nanpercentile(values, [15.865, 84.135], axis=0),
    }


In [ ]:
R = Retrieval.load(RETRIEVAL_PATH)
alpha = float(getattr(R, "alpha", 0))
ROUND_INDEX = resolve_round(OUTPUT_DIR, ROUND)
data_path = round_data_path(OUTPUT_DIR, ROUND_INDEX)
round_data = RetrievalOutput.load_round_data(data_path)
spectra = round_data.get("post_spec", round_data["spec"])
mask = proposal_sample_mask(round_data, alpha)

print(f"Loaded retrieval: {RETRIEVAL_PATH}")
print(f"Selected round: {ROUND_INDEX}")
print(f"Loaded round data: {data_path}")
print(f"alpha = {alpha}")
if isinstance(mask, np.ndarray):
    print(f"Using {mask.sum()} proposal-sampled samples out of {len(mask)} total samples")


## Corner Plot


In [ ]:
proposal = proposal_for_corner(R, alpha)
samples = proposal.sample((N_CORNER_SAMPLES,)).detach().cpu().numpy()
samples = samples.reshape(-1, len(R.parameters))
samples_physical = helpers.convert_cube(samples, R.parameters)

fig = corner(samples_physical, labels=list(R.parameters.keys()), show_titles=True)
fig.suptitle("Uninflated posterior" if alpha > 0 else "Posterior", y=1.02)
plt.show()


## Retrieved Spectra


In [ ]:
obs_items = list(R.obs.items())
fig, axes = plt.subplots(len(obs_items), 1, figsize=(16, 5 * len(obs_items)), squeeze=False)



for ax, (obs_key, obs) in zip(axes[:, 0], obs_items):
    spec = spectra[obs_key][mask]
    median = np.nanmedian(spec, axis=0)
    bands = sigma_bands(spec)
    for name, alpha_fill in [("3sigma", 0.10), ("2sigma", 0.16), ("1sigma", 0.26)]:
        lo, hi = bands[name]
        ax.fill_between(obs[:, 0], lo, hi, color="tab:blue", alpha=alpha_fill, lw=0, label=name)
    ax.plot(obs[:, 0], median, color="tab:blue", lw=2, label="median model")
    ax.errorbar(obs[:, 0], obs[:, 1], yerr=obs[:, 2], fmt="-", ms=3, color="black", label="data")
    ax.set_title(str(obs_key))
    ax.set_xlabel("Wavelength")
    ax.set_ylabel("Flux / depth")
    ax.legend()
    ax.set_ylim(-3e-7, 2e-5)

fig.tight_layout()
plt.show()


## Atmospheric Structures


In [ ]:
def parse_metadata(line):
    metadata = {}
    for item in line.lstrip("#").split():
        if "=" in item:
            key, value = item.split("=", 1)
            metadata[key] = value
    return metadata


def parse_columns(line):
    return line.split("=", 1)[1].split()


def parse_mixingratios(path):
    blocks = []
    current = []
    for line in Path(path).read_text().splitlines():
        if line.startswith("# columns=") and current:
            blocks.append(current)
            current = []
        current.append(line)
    if current:
        blocks.append(current)

    profiles = []
    for block in blocks:
        columns = None
        metadata = {}
        numeric = []
        for line in block:
            stripped = line.strip()
            if not stripped:
                continue
            if stripped.startswith("# columns="):
                columns = parse_columns(stripped)
                continue
            if stripped.startswith("# round="):
                metadata = parse_metadata(stripped)
                continue
            if stripped.startswith("#"):
                continue
            numeric.append(stripped)
        if not numeric:
            continue
        arr = np.loadtxt(StringIO("\n".join(numeric)))
        arr = np.atleast_2d(arr)
        if columns is None:
            raise ValueError(
                f"No '# columns=' metadata found for an atmosphere block in {path}. "
                "Re-run ARCiS with the current FlopPITy branch so the header is preserved."
            )
        if len(columns) != arr.shape[1]:
            raise ValueError(f"Column count mismatch in {path}: {len(columns)} names for {arr.shape[1]} columns")
        profiles.append({
            "columns": columns,
            "metadata": metadata,
            "data": {name: arr[:, i] for i, name in enumerate(columns)},
        })
    return profiles


def filter_profiles_by_source(profiles, round_data, alpha):
    sources = round_data.get("sample_sources")
    if alpha <= 0:
        return profiles
    if sources is None:
        raise ValueError("This run used alpha > 0, but training_data.npz has no sample_sources.")
    selected = []
    for profile in profiles:
        global_model = int(profile["metadata"].get("global_model", -1))
        if 0 <= global_model < len(sources) and sources[global_model] == "proposal":
            selected.append(profile)
    return selected


def profile_grid(profiles, value_name, log_value=False, n_grid=200):
    ranges = [
        (np.nanmin(p["data"]["P"]), np.nanmax(p["data"]["P"]))
        for p in profiles
        if value_name in p["data"]
    ]
    if not ranges:
        raise KeyError(f"No profiles contain column {value_name!r}")
    p_min = max(low for low, high in ranges)
    p_max = min(high for low, high in ranges)
    pressure = np.geomspace(p_min, p_max, n_grid)
    log_pressure = np.log10(pressure)
    values = []
    for profile in profiles:
        data = profile["data"]
        if value_name not in data:
            continue
        p = np.asarray(data["P"])
        v = np.asarray(data[value_name])
        order = np.argsort(p)
        y = np.log10(np.clip(v[order], 1e-300, None)) if log_value else v[order]
        values.append(np.interp(log_pressure, np.log10(p[order]), y))
    return pressure, np.asarray(values)


def component_label(profile):
    thread = profile["metadata"].get("thread", "")
    marker = "_component"
    if marker not in thread:
        return "component 1"
    suffix = thread.split(marker, 1)[1]
    digits = "".join(char for char in suffix if char.isdigit())
    return f"component {digits or suffix}"


def component_sort_key(label):
    digits = "".join(char for char in label if char.isdigit())
    return int(digits) if digits else label


def profiles_by_component(profiles):
    groups = {}
    for profile in profiles:
        groups.setdefault(component_label(profile), []).append(profile)
    return dict(sorted(groups.items(), key=lambda item: component_sort_key(item[0])))


def component_subplots(n_components, width=5, height=6, layout="horizontal"):
    if layout == "vertical":
        fig, axes = plt.subplots(
            n_components,
            1,
            figsize=(width, height * n_components),
            squeeze=False,
            sharey=True,
        )
    else:
        fig, axes = plt.subplots(
            1,
            n_components,
            figsize=(width * n_components, height),
            squeeze=False,
            sharey=True,
        )
    return fig, axes.ravel()


def orient_pressure_axis(axes, pressure):
    pressure = np.asarray(pressure)
    for ax in np.ravel(axes):
        ax.set_yscale("log")
        ax.set_ylim(np.nanmax(pressure), np.nanmin(pressure))


atm_path = atmosphere_path(OUTPUT_DIR, ROUND_INDEX)
profiles_all = parse_mixingratios(atm_path)
profiles = filter_profiles_by_source(profiles_all, round_data, alpha)
component_profiles = profiles_by_component(profiles)
columns = profiles[0]["columns"]
species = [name for name in columns if name not in {"T", "P", "H2", "He"}]

print(f"Loaded {len(profiles)} atmospheric profiles from {atm_path}")
print(f"Components: {list(component_profiles)}")
print(f"Species: {species}")


## Temperature Structure


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 7))
component_colors = plt.cm.tab10(np.linspace(0, 1, max(len(component_profiles), 1)))
last_pressure = None

for color, (component_name, component_group) in zip(component_colors, component_profiles.items()):
    pressure, temperature = profile_grid(component_group, "T", log_value=False)
    last_pressure = pressure
    bands = sigma_bands(temperature)
    for name, alpha_fill in [("3sigma", 0.08), ("2sigma", 0.13), ("1sigma", 0.22)]:
        lo, hi = bands[name]
        ax.fill_betweenx(pressure, lo, hi, color=color, alpha=alpha_fill, lw=0)
    ax.plot(
        np.nanmedian(temperature, axis=0),
        pressure,
        color=color,
        lw=2,
        label=f"{component_name} median T",
    )

orient_pressure_axis([ax], last_pressure)
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Pressure [bar]")
ax.legend()
fig.tight_layout()
plt.show()


## Molecular Abundances


In [ ]:
fig, axes = component_subplots(len(component_profiles), width=7, height=4.2, layout="vertical")
last_pressure = None
legend_handles = []
legend_labels = []

for ax, (component_name, component_group) in zip(axes, component_profiles.items()):
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(species), 1)))
    for color, species_name in zip(colors, species):
        pressure, abundance = profile_grid(component_group, species_name, log_value=True)
        last_pressure = pressure
        lo, hi = sigma_bands(abundance)["1sigma"]
        median = np.nanmedian(abundance, axis=0)
        ax.fill_betweenx(pressure, lo, hi, color=color, alpha=0.16, lw=0)
        line, = ax.plot(median, pressure, color=color, lw=2, label=species_name)
        if species_name not in legend_labels:
            legend_handles.append(line)
            legend_labels.append(species_name)

    ax.set_title(component_name)
    ax.set_xlabel("log10 abundance")

orient_pressure_axis(axes, last_pressure)
for ax in axes:
    ax.set_ylabel("Pressure [bar]")

fig.legend(
    legend_handles,
    legend_labels,
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
)
fig.subplots_adjust(hspace=0.0, right=0.78)
plt.show()
